In [0]:
import pypsa

In [0]:
%sh

set -euo pipefail

export PATH="$HOME/.pixi/bin:$PATH"

if ! command -v pixi >/dev/null 2>&1; then
  curl -fsSL https://pixi.sh/install.sh | sh
fi

SRC="/Workspace/Repos/luisa.kamp@oge.net/pypsa-de-test/"
DST="/local_disk0/pypsa-de-test/"

mkdir -p "$DST"

rsync -a \
  --exclude=".pixi/" \
  --exclude="scripts_LuKa/04_TEST" \
  "$SRC" "$DST"

cd "$DST"

pixi install

pixi run snakemake --version

PYTHONPATH="" pixi run snakemake -c1 build_scenarios

### Durchführung von Optimierung inkl. Auswertung und Plots (für ein Stützjahr)

In [0]:
# %sh

# export PATH="$HOME/.pixi/bin:$PATH"

# cd /local_disk0/pypsa-de-v04-SR27-3

# # snakemake -c1 all

# PYTHONPATH="" pixi run snakemake -n -p -c1 all \
#   --config planning_horizons='[2025]'

### Durchführung nur Optimierung

In [0]:
# %sh

# export PATH="$HOME/.pixi/bin:$PATH"

# cd /local_disk0/pypsa-de-test

# # PYTHONPATH="" pixi run snakemake -c8 --jobs 1 \
# #   results/20261005_restructure_workflow/KN2045_Mix/networks/solved_2025.nc \
# #   --config planning_horizons='[2025]' \
# #   --resources mem_mb=60000 \
# #   --set-resources solve_network:mem_mb=60000

# # PYTHONPATH="" pixi run snakemake -n -p -c8 --jobs 1 \     # DRYRUN
# PYTHONPATH="" pixi run snakemake -c8 --jobs 1 \
#   results/20260422_small_changes/KN2045_Mix/networks/base_s_27__none_2025.nc \
#   --forcerun modify_prenetwork solve_sector_network_myopic \
#   --resources mem_mb=60000 \
#   --set-resources solve_sector_network_myopic:mem_mb=60000

### Durchführung Optimierung aller Jahre

In [0]:
%sh
set -euo pipefail

export PATH="$HOME/.pixi/bin:$PATH"
cd /local_disk0/pypsa-de-test

RESULTS="results/20260422_small_changes/KN2045_Mix/networks"
BACKUP="/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/original"

mkdir -p "$BACKUP"

for YEAR in 2025 2030 2035 2040 2045 2050; do
  TARGET="$RESULTS/base_s_27__none_${YEAR}.nc"

  echo "=== Optimierung $YEAR ==="

  PYTHONPATH="" pixi run snakemake -c8 --jobs 1 \
    "$TARGET" \
    --resources mem_mb=60000 \
    --set-resources solve_sector_network_myopic:mem_mb=60000

  cp "$TARGET" "$BACKUP/"
  cmp "$TARGET" "$BACKUP/$(basename "$TARGET")"

  echo "=== $YEAR erfolgreich gespeichert und geprüft ==="
done

### Speichern auf Volume

In [0]:
# %sh
# set -euo pipefail

# SRC="/local_disk0/pypsa-de-test/results/20260422_small_changes/KN2045_Mix/networks/base_s_27__none_2025.nc"
# DST="/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/original"

# mkdir -p "$DST"
# cp "$SRC" "$DST/"

# cmp "$SRC" "$DST/$(basename "$SRC")"
# echo "Netzwerk erfolgreich kopiert und geprüft."
# ls -lh "$DST/$(basename "$SRC")"

In [0]:
# %sh

# mkdir -p /Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/

# cp -r \
#   /local_disk0/pypsa-de-v04-SR27-3/results/20260422_small_changes \
#   /Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/